In [1]:
from math import sqrt, ceil
from statistics import NormalDist

N = NormalDist()


def n_per_arm(p1, p2, alpha=0.05, power=0.80):
    """So nguoi moi nhom (cong thuc pooled-variance duoi H0)."""
    za = N.inv_cdf(1 - alpha / 2)
    zb = N.inv_cdf(power)
    pbar = (p1 + p2) / 2
    num = (za * sqrt(2 * pbar * (1 - pbar)) + zb * sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2
    return num / (p2 - p1) ** 2


def power_at_n(p1, p2, n, alpha=0.05):
    """Power thuc te khi co n nguoi moi nhom."""
    za = N.inv_cdf(1 - alpha / 2)
    se = sqrt(p1 * (1 - p1) / n + p2 * (1 - p2) / n)
    return N.cdf(abs(p2 - p1) / se - za)


if __name__ == "__main__":
    P1, UPLIFT, ALPHA, POWER = 0.20, 0.05, 0.05, 0.80
    P2 = P1 + UPLIFT

    base = n_per_arm(P1, P2, ALPHA, POWER)
    print("=== KET QUA CHINH ===")
    print(f"Baseline Control p1 = {P1:.0%}, Treatment p2 = {P2:.0%} (uplift {UPLIFT*100:.0f} diem %)")
    print(f"alpha = {ALPHA} (hai phia), power = {POWER:.0%}, ty le 1:1")
    print(f"n moi nhom (chua hieu chinh) = {base:.1f} -> lam tron len {ceil(base)}")

    print("\n=== HIEU CHINH HAO HUT / HIEU UNG CUM ===")
    for loss in (0.05, 0.10):
        print(f"Hao hut {loss:.0%}: n = {ceil(base / (1 - loss))}")
    for deff in (1.05, 1.10):
        print(f"Design effect {deff}: n = {ceil(base * deff)}")

    print("\n=== KIEM TRA n = 1200 ===")
    print(f"Power thuc te voi 1200/nhom: {power_at_n(P1, P2, 1200):.1%}")
    for d in (0.03, 0.04, 0.045, 0.05):
        print(f"  Neu uplift that = {d*100:.1f} diem %: power = {power_at_n(P1, P1 + d, 1200):.1%}")

    print("\n=== CANH BAO: neu '5%' la tang TUONG DOI (20% -> 21%) ===")
    print(f"n moi nhom = {ceil(n_per_arm(0.20, 0.21))}")

    print("\n=== BANG DO NHAY: n moi nhom (power 80%) ===")
    ups = (0.03, 0.04, 0.05, 0.06)
    print("baseline | " + " | ".join(f"+{u*100:.0f}pp" for u in ups))
    for b in (0.15, 0.20, 0.25):
        print(f"{b:>8.0%} | " + " | ".join(f"{ceil(n_per_arm(b, b+u)):>5}" for u in ups))

    print("\n=== BANG DO NHAY: n moi nhom (power 90%) ===")
    print("baseline | " + " | ".join(f"+{u*100:.0f}pp" for u in ups))
    for b in (0.15, 0.20, 0.25):
        print(f"{b:>8.0%} | " + " | ".join(f"{ceil(n_per_arm(b, b+u, power=0.9)):>5}" for u in ups))

    print("\n=== THOI GIAN TUYEN ===")
    total = 2 * 1200
    print(f"Tong can {total} nguoi; voi 3000 nguoi du dieu kien/thang -> ~{total/3000*30:.0f} ngay")

=== KET QUA CHINH ===
Baseline Control p1 = 20%, Treatment p2 = 25% (uplift 5 diem %)
alpha = 0.05 (hai phia), power = 80%, ty le 1:1
n moi nhom (chua hieu chinh) = 1093.7 -> lam tron len 1094

=== HIEU CHINH HAO HUT / HIEU UNG CUM ===
Hao hut 5%: n = 1152
Hao hut 10%: n = 1216
Design effect 1.05: n = 1149
Design effect 1.1: n = 1204

=== KIEM TRA n = 1200 ===
Power thuc te voi 1200/nhom: 83.6%
  Neu uplift that = 3.0 diem %: power = 43.2%
  Neu uplift that = 4.0 diem %: power = 65.8%
  Neu uplift that = 4.5 diem %: power = 75.6%
  Neu uplift that = 5.0 diem %: power = 83.6%

=== CANH BAO: neu '5%' la tang TUONG DOI (20% -> 21%) ===
n moi nhom = 25583

=== BANG DO NHAY: n moi nhom (power 80%) ===
baseline | +3pp | +4pp | +5pp | +6pp
     15% |  2402 |  1384 |   906 |   643
     20% |  2943 |  1683 |  1094 |   772
     25% |  3397 |  1933 |  1251 |   878

=== BANG DO NHAY: n moi nhom (power 90%) ===
baseline | +3pp | +4pp | +5pp | +6pp
     15% |  3215 |  1852 |  1212 |   860
     20% |